# KWS training — Speech Commands v2 → DNN / CNN / DS-CNN → INT8

Runtime → Change runtime type → **T4 GPU**. Then *Run all*.

Steps: clone the repo, download Speech Commands v2 (~2.3 GB), build the MFCC cache, train the three models, quantize them to INT8 and download the artifacts.
The feature cache and artifacts are also saved to Google Drive so a disconnected session can resume.

In [ ]:
BRANCH = "feat/training"   # use "main" once the branch is merged
USE_DRIVE = True            # keep the feature cache and artifacts in Drive
EPOCHS = 40
MODELS = ["dnn", "cnn", "dscnn"]

!nvidia-smi -L || echo "no GPU: training will be slow"

In [ ]:
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/kws-freertos-esp32s3"
else:
    WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)

!rm -rf /content/repo && git clone -q -b {BRANCH} https://github.com/danielopezr/kws-freertos-esp32s3.git /content/repo
%cd /content/repo/training
!git log --oneline -1

In [ ]:
# Unit tests of the MFCC front end and the dataset builder (numpy only, ~5 s)
!python -m unittest discover -s tests

In [ ]:
# Raw audio stays on the local disk (fast); the feature cache goes to WORK.
CACHE = f"{WORK}/features"
if os.path.exists(f"{CACHE}/meta.json"):
    print("feature cache found, skipping download")
else:
    !python prepare_data.py --data-dir /content/speech_commands --cache-dir {CACHE}

In [ ]:
ART = f"{WORK}/artifacts"
for m in MODELS:
    !python train.py --model {m} --cache-dir {CACHE} --out {ART} --epochs {EPOCHS}
    !python quantize.py --model {m} --cache-dir {CACHE} --out {ART}

In [ ]:
import json, pathlib
rows = []
for m in MODELS:
    tr = json.loads(pathlib.Path(f"{ART}/{m}/train_report.json").read_text())
    q = json.loads(pathlib.Path(f"{ART}/{m}/quant_report.json").read_text())
    rows.append((m, tr["params"], q["tflite_bytes"] / 1024, q["test_accuracy_float"], q["test_accuracy_int8"], q["test_macro_f1_int8"]))
print(f"{'model':8}{'params':>10}{'tflite KB':>11}{'acc float':>11}{'acc int8':>10}{'F1 int8':>9}")
for m, p, kb, af, ai, f1 in rows:
    print(f"{m:8}{p:>10,}{kb:>11.1f}{af:>11.4f}{ai:>10.4f}{f1:>9.4f}")

In [ ]:
# Download everything except the Keras checkpoints (they are only needed to re-quantize)
!cd {ART} && zip -qr /content/kws_artifacts.zip . -x "*.keras"
from google.colab import files
files.download("/content/kws_artifacts.zip")